# Phase 4: Literature Validation — ML vs Mayo Baseline

对比SimpViT ML模型与传统Mayo ODE拟合在77个已发表Ctr值上的预测精度。
同时对inhibition period和retardation factor进行定性验证。

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.join("..", "src"))

from literature_validation import (
    run_validation_pipeline, compute_summary_stats,
    fold_error_log, fold_error_ratio, RAFT_COLORS
)
print("Imports OK")

In [ ]:
CSV_PATH = "../data/literature/literature_ctr.csv"
MODEL_PATH = "../checkpoints/best_model.pth"
BOOTSTRAP_PATH = None  # "../checkpoints/bootstrap_heads.pth" if available
CALIBRATION_PATH = None  # "../checkpoints/calibration_factors.json" if available
OUTPUT_DIR = "../figures/validation"
SIGMA = 0.03
SEED = 42
print(f"Model: {MODEL_PATH}")
print(f"CSV: {CSV_PATH}")

In [ ]:
results_df, summary = run_validation_pipeline(
    CSV_PATH, MODEL_PATH,
    bootstrap_path=BOOTSTRAP_PATH,
    calibration_path=CALIBRATION_PATH,
    output_dir=OUTPUT_DIR,
    sigma=SIGMA,
    device="cpu",
    seed=SEED
)
print(f"Validated {len(results_df)} literature points")

In [ ]:
display_cols = ["id", "raft_type", "method", "log10_ctr_true",
               "ml_log10_ctr", "mayo_log10_ctr",
               "ml_fold_error", "mayo_fold_error",
               "ml_inhibition", "ml_retardation"]
pd.set_option("display.float_format", "{:.3f}".format)
results_df[display_cols]

In [ ]:
print("=== ML Summary ===")
print(json.dumps(summary["ml"], indent=2))
print("
=== Mayo Summary ===")
print(json.dumps(summary["mayo"], indent=2))

In [ ]:
from IPython.display import Image
Image(filename=os.path.join(OUTPUT_DIR, "parity_ml_vs_mayo.png"), width=600)

## Per-RAFT-Type Fold-Error Breakdown

按RAFT剂类型分组报告ML和Mayo的中位fold-error。

In [ ]:
for raft_type in ["dithioester", "trithiocarbonate", "xanthate", "dithiocarbamate"]:
    sub = results_df[results_df["raft_type"] == raft_type]
    if sub.empty:
        continue
    ml_med = sub["ml_fold_error"].median()
    mayo_med = sub["mayo_fold_error"].median()
    print(f"{raft_type:20s}  n={len(sub)}  ML median fold-error: {ml_med:.2f}x  Mayo: {mayo_med:.2f}x")

In [ ]:
print("\n=== Inhibition Period & Retardation Factor (Qualitative) ===\n")
print("Expected: dithioester → elevated inhibition, retardation < 1.0")
print("          TTC/xanthate/dithiocarbamate → inhibition ≈ 0, retardation ≈ 1.0\n")
for raft_type in ["dithioester", "trithiocarbonate", "xanthate", "dithiocarbamate"]:
    sub = results_df[results_df["raft_type"] == raft_type]
    if sub.empty:
        continue
    inh_mean = sub["ml_inhibition"].mean()
    inh_std = sub["ml_inhibition"].std()
    ret_mean = sub["ml_retardation"].mean()
    ret_std = sub["ml_retardation"].std()
    print(f"{raft_type:20s}  n={len(sub):2d}  "
          f"inhibition={inh_mean:.3f}±{inh_std:.3f}  "
          f"retardation={ret_mean:.3f}±{ret_std:.3f}")

In [ ]:
from IPython.display import Image
Image(filename=os.path.join(OUTPUT_DIR, "inhibition_retardation_by_class.png"), width=800)

## Notes

**扩展数据集:** 77个文献Ctr值，覆盖4种RAFT剂类型（dithioester, trithiocarbonate, xanthate, dithiocarbamate），来源于Chong 2003, Moad 2009, Moad 2012。

**公平对比设计 (D-07):** ML和Mayo均在相同的ODE模拟数据上运行（加σ=0.03噪声），消除真实实验数据异质性的干扰。

**Mayo有利条件 (D-06):** Mayo拟合时其他动力学参数固定为训练分布中心值，这对Mayo是有利的（理想参数）。

**Bootstrap CI:** 当前无bootstrap检查点，CI使用集成std代替。训练bootstrap后可重新运行以获得95%置信区间。

**Inhibition/Retardation定性验证:** 文献中无定量数值，采用定性验证——检查模型预测是否符合已知RAFT化学行为：
- Dithioester: 应有明显诱导期（slow pre-equilibrium fragmentation），减速因子显著 < 1.0
- TTC/xanthate/dithiocarbamate: 诱导期≈0，减速因子≈1.0（理想RAFT交换）